<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Your Places, Week 7 Session 2: What Falls Inside What

### CP101: Introduction to Urban Data Analytics
#### Wednesday, GeoPandas III

On Monday you drew 500 m circles, asked a predicate what they touched, and ran a first spatial
join against 398 parks. That join handed back **456 rows from 398 parks** and the notebook left
the problem there.

Today you pick it up at a scale where it matters, then go past the join altogether: **overlay**
cuts new shapes rather than matching rows, and the last section is about everything the join
could not find.

**One note on the data.** `class_places.csv` holds 631 distinct places, pooled and anonymised
from a **previous** CP101 class rather than your own. They are real submissions from real
students, which is worth remembering in Week 13 when you are asked whether you would publish a
map of them. Its `times_listed` and `access_score` are real; `share_loved` comes from the
generated `rating` column.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 1. Set up, without naming any place in the code

Your places file is yours to edit, so nothing below may depend on a particular place being in
it. Take the anchor by **position** and print which one it turned out to be.

*What happens: a place name and a count. Before you run it, decide which of your places will be chosen as the anchor.*

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="white")

places_df = pd.read_csv("../data/raw/my_places.csv")
places = gpd.GeoDataFrame(
    places_df,
    geometry=gpd.points_from_xy(x=places_df["lon"], y=places_df["lat"]),
    crs=4326,
)
metres = places.estimate_utm_crs()
places = places.to_crs(metres)

anchor_name = places["name"].iloc[0]      # the first row, whatever it is
anchor = places[places["name"] == anchor_name]

print("anchor:", anchor_name)
print("my places:", len(places))

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 2. A predicate gives you booleans

A quick reprise of Monday, because everything below builds on it. `.within()` asks a yes-or-no
question of every row: is this point inside that shape? What comes back is a column of `True`
and `False`, one per row, the same kind of mask you built in Week 5, made by geometry instead of
by a comparison.

*What happens: your places with a `True`/`False` column. Before you run it, decide whether the anchor itself will come back `True`.*

In [ ]:
anchor_zone = anchor.buffer(500).iloc[0]

places["within_500m"] = places.within(anchor_zone)
places[["name", "category", "within_500m"]]

The anchor is inside its own circle, so its row is always `True` and tells you nothing.
Two of your other places are inside and one is not, and the one that is not is about 525 m away,
which is a reminder that 500 m is a line somebody drew, not a fact about the campus.

A predicate answers *whether*. It cannot answer *which zone*, or *what else is true about that
zone*, because a column of `True` and `False` has nowhere to put that.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: drop the useless row

1. **Drop the useless row.** The anchor is always inside its own circle. Show the section 2 table
   without that row.

<details>
<summary><b>Check your answer</b></summary>

The anchor is inside its own circle, so it always matches. Removing it is a filter on the name, and it is worth doing because a self-match is not a finding.

</details>

In [ ]:
# Your code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 3. A spatial join gives you the other table's columns

`gpd.sjoin` asks the same geometric question and hands back **matched rows plus the attributes of
whatever they matched**. That is the whole difference, and it is why the join is the tool you
reach for as soon as there is more than one zone.

Build one zone per place first, so there are four shapes to be matched against.

*What happens: four zone polygons, each carrying the name of the place at its centre. Before you run it, decide how many rows this table will have.*

In [ ]:
# [["name"]] with DOUBLE brackets keeps this a TABLE that happens to have one column.
# ["name"] with single brackets would hand back the column on its own, with nowhere to
# put a geometry. .copy() means later edits to zones cannot reach back into places.
zones = places[["name"]].copy()

zones["geometry"] = places.buffer(500)             # a 500 m circle around each place

# a GeoDataFrame has to be told which column holds the shapes, and in what CRS
zones = gpd.GeoDataFrame(zones, geometry="geometry", crs=metres)

zones = zones.rename(columns={"name": "zone"})     # rename now, so the join has no clash later

zones

Now the class file: 631 places listed by 77 students. Which of them fall inside one of
your four zones, and which zone?

*What happens: a row count for the class file, then a row count after the join. Before you run it, decide whether the join will return more rows or fewer than 631.*

In [ ]:
class_df = pd.read_csv("../data/processed/class_places.csv")
class_places = gpd.GeoDataFrame(
    class_df,
    geometry=gpd.points_from_xy(x=class_df["lon"], y=class_df["lat"]),
    crs=4326,
).to_crs(metres)

print("class places:", len(class_places))

reached = gpd.sjoin(class_places, zones, predicate="within")
print("rows after the join:", len(reached))
print("distinct class places reached:", reached["name"].nunique())

reached[["name", "category", "times_listed", "zone"]].head(8)

🔍 **The join returned 794 rows from a table of 631.** That is Monday's 398-into-456 again,
at a scale where it is harder to shrug off, and it is the single most important thing to
understand about spatial joins.

`sjoin` returns **one row per match**. A class place that falls inside two of your circles matches
twice, so it appears twice, once labelled with each zone. Only 312 distinct places were reached,
but they generated 794 rows between them, because your zones overlap.

If you had counted `len(reached)` and called it "places near me", you would have been wrong by
more than double, and nothing would have told you.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 4. Collapsing the explosion

`groupby` is what turns many rows back into one answer per thing you care about. Which of your
places has the most of the class's places around it?

*What happens: four rows, one per zone, with a count and a rate. Before you run it, decide which of your places will have the most class places within 500 m.*

In [ ]:
summary = reached.groupby("zone").agg(
    class_places_within=("name", "size"),          # a count of matched rows
    listings_within=("times_listed", "sum"),       # how many student listings that represents
    mean_access=("access_score", "mean"),          # a rate, averaged
).reset_index()
summary["mean_access"] = summary["mean_access"].round(3)
summary = summary.sort_values("class_places_within", ascending=False)
summary

Those four numbers are the first real planning answer this project has produced. **In
the file as shipped**, they say that the corner of campus around Yogurt Park has roughly 50 per
cent more of the class's places within 500 m **as the crow flies** than the corner around
Memorial Glade does. Edit your own places file and both the names and the ratio change, which is
exactly why section 1 refused to write any place name into the code.

Notice what "500 m as the crow flies" is not. It is not a five-minute walk. Nobody walks through
buildings, and in Week 13 you measure the real distance across the street network for two of
these places and find it about 46 per cent longer than the straight line. We find out there what
a circle is worth.

Read `mean_access` carefully too. It is the average of an average: each place's `access_score` is
already a rate across its own listings, and this averages those rates, so a stairwell one student
mentioned counts exactly as much as the cafe forty-one students mentioned. That may be what you
want. It is not the same as the share of all listings in the zone that said `yes`, and the two
will disagree. Going deeper computes the other one.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: change the anchor

1. **Change the anchor.** Set `anchor_name` to a different one of your places by changing
   `.iloc[0]` to `.iloc[2]`, and re-run section 2. If A is within 500 m of B, is B within 500 m
   of A? Say why that has to be true for a circle, and name a shape for which it would not be.

<details>
<summary><b>Check your answer</b></summary>

A different anchor gives a different set of matched places, and usually a different count. If the count did not change at all, check that you changed the name that the rest of the notebook actually reads.

</details>

In [ ]:
# Your code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 5. Overlay: making new shapes, not matching rows

The join in section 3 told you **which** class places fall inside a zone. Look at what it
did not do: `zones` still holds four separate circles, overlapping in the middle, exactly
as they were. A join matches rows. It never changes a shape.

So there are questions the join cannot answer at all:

- How much ground do my four zones cover **together**?
- How much of that ground is actually built on?

Both need *new geometry*. That is what a **spatial operation** does, and it is the other
half of this session.

### Adding up circles counts the overlap twice

Monday's optional section already measured this, and if you skipped it, here it is: four
circles of the same size do **not** cover four circles' worth of ground once they touch, and
yours touch a lot. What matters today is the *tool*. Monday used `union_all()`, which returns
one bare shape. `dissolve()` does the same merge but hands back a **GeoDataFrame**, which is
what the overlay in the next section needs as an input.

In [ ]:
naive_total = zones.geometry.area.sum() / 1_000_000

# dissolve() with no "by" column fuses every row into one shape, and the overlaps
# disappear into it. This is a spatial OPERATION: the output geometry did not exist before.
served = zones.dissolve().reset_index(drop=True)[["geometry"]]
served_total = served.geometry.area.sum() / 1_000_000

print(f"four circles, added up:  {naive_total:.3f} km2")
print(f"the same ground, once:   {served_total:.3f} km2")
print(f"counted twice:           {naive_total - served_total:.3f} km2 "
      f"({(naive_total - served_total) / naive_total:.0%} of the naive total)")

*What happens: the four circles add up to about 3.14 km², but the ground they
actually cover is about 1.47 km². **Over half the naive total is the same ground counted
more than once.** If you had reported "my places serve 3.1 km² of Berkeley" you would have
been wrong by a factor of two, and nothing in the data would have told you.*

This is the same error as a bar chart that double counts a category, and it is much harder
to see, because area is not something the eye adds up.

### `overlay` cuts one layer by another

`dissolve` combined shapes within one layer. **`overlay` works across two layers**, and
`how="intersection"` keeps only the ground the two have in common, cutting new polygons at
the boundary where they meet.

Ask it something a predicate cannot answer: of the ground I claim to serve, how much is
building footprint rather than street, yard or open space?

In [ ]:
buildings = gpd.read_parquet("../data/raw/buildings.parquet").to_crs(metres)

built_inside = gpd.overlay(
    served,
    buildings[["geometry"]],
    how="intersection",
    keep_geom_type=True,     # keep polygons only; a shared edge can yield a line
)

built_km2 = built_inside.geometry.area.sum() / 1_000_000

print(f"buildings in the file:        {len(buildings)}")
print(f"pieces after the overlay:     {len(built_inside)}")
print(f"building footprint inside:    {built_km2:.3f} km2")
print(f"share of the served area:     {built_km2 / served_total:.1%}")

*What happens: about a third of the ground inside your 500 m zones is building
footprint. The overlay produced hundreds of new polygons, one for each building clipped
against the served area, and none of them existed in either input file.*

Note the difference from section 3. `sjoin` would have told you **which** buildings are in
range and handed back whole footprints, including the parts sticking out past the edge of
your zone. `overlay` cut them at the edge and gave you the part that is genuinely inside.
When you are counting things, the join is right. When you are measuring ground, only the
overlay is.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: which tool answers which question

1. **Say which one.** For each question below, name the tool: a predicate, `sjoin`,
   `dissolve`, or `overlay`.
   *(a)* Which of my places are within 500 m of a park?
   *(b)* How many square metres of park are inside my served area?
   *(c)* How much ground do my zones cover in total?
2. **Do one of them.** Compute (b) for `parks.parquet`, the same way the buildings cell
   works. Then compare it with the whole area of every park that merely *touches* your
   served area, using `parks[parks.intersects(served.geometry.iloc[0])]`. How different are
   the two numbers?

<details>
<summary><b>Check your answer</b></summary>

*(a)* a predicate, or `sjoin` if you also want the park's columns. You are asking which
rows qualify, and no shape needs to change.
*(b)* `overlay`. You are measuring ground, so the parks have to be cut at the edge of the
served area.
*(c)* `dissolve` (or `union_all`), because the circles overlap and must be fused first.

For 2, the two numbers come out close here, within about 10%, because Berkeley's mapped
parks are small and most of the ones near your places sit entirely inside the zone. Do not
read that as "the difference does not matter". It is small **because of this data**. In
Lab 7 you do exactly this against census tracts, which are far larger than a 500 m circle,
and there the whole-tract number overstates the answer enormously.

</details>

In [ ]:
# Your code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 6. Missing data, and the three ways it hides

⏱ **Time check. If it is past 45 minutes, run this section's cells and read the bold lines
only.** This section is short on code and long on consequences, and the `.dropna()` result below
is the one to leave with.

Every table so far has been complete enough to ignore the question. `parks.parquet` is not, and
it is the ordinary case: **369 of its 398 parks have no name at all.**

The word "missing" is doing three different jobs, and they need different answers.

*What happens: a count of nulls per column, then the same for the parks file. Before you run it, decide what share of parks you expect to be unnamed.*

In [ ]:
parks = gpd.read_parquet("../data/raw/parks.parquet").to_crs(metres)

print("parks:", len(parks))
print()
print("nulls per column:")
print(parks.isna().sum())
print()
print("share of parks with no name:", f"{parks['name'].isna().mean():.1%}")
print("distinct named parks:", parks["name"].nunique())

**Kind one: the attribute is missing and the row is fine.** These 369 parks are real
polygons in real places. Only the label is absent, because nobody in OpenStreetMap typed one.
Their geometry is perfectly good and it is the geometry you are about to measure.

Which sets up the trap. The instinct when a column is 93 per cent empty is to clean it away.

*What happens: the same overlay measured twice, once on all parks and once after dropping the unnamed ones. Before you run it, decide how much park area the second measurement will lose.*

In [ ]:
# the honest measurement: cut every park to the served area, then add up the pieces
park_in_zone = gpd.overlay(served, parks[["name", "geometry"]],
                           how="intersection", keep_geom_type=True)
all_km2 = park_in_zone.area.sum() / 1_000_000

# the tidy-looking one: drop the rows with no name FIRST, then measure
named_only = parks.dropna(subset=["name"])
named_in_zone = gpd.overlay(served, named_only[["name", "geometry"]],
                            how="intersection", keep_geom_type=True)
named_km2 = named_in_zone.area.sum() / 1_000_000

print(f"park area inside my zones, all parks   : {all_km2:.4f} km2  ({len(park_in_zone)} pieces)")
print(f"park area inside my zones, named only  : {named_km2:.4f} km2  ({len(named_in_zone)} pieces)")
print(f"lost by dropping unnamed parks         : {all_km2 - named_km2:.4f} km2")
print(f"the tidy version understates by        : {1 - named_km2 / all_km2:.1%}")

🔍 **`.dropna(subset=["name"])` threw away 60 per cent of the park.**

Nothing about that line looks dangerous. It removes rows with a missing label, which is a
reasonable thing to do to a column you are about to display. But the rows it removed were
carrying **geometry you were measuring**, and area does not care what a park is called.

This is the difference between `pandas` and `geopandas` that matters most. In a plain table a
row with a missing label is a row you probably cannot use. In a spatial table, the label and the
shape are independent, and **dropping on a missing attribute silently deletes ground.**

*What happens: the two measurements side by side. Before you run it, decide which bar you would have published.*

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 2.6), constrained_layout=True)

# barh draws the first item at the bottom, so list them in reverse reading order
labels = ['after .dropna(subset=["name"])', "every park\n(the honest answer)"]
values = [named_km2, all_km2]
colors = ["#c05621", "#8A9A5B"]

bars = ax.barh(labels, values, color=colors, height=0.58, zorder=3)
for bar, v in zip(bars, values):
    ax.text(v + all_km2 * 0.02, bar.get_y() + bar.get_height() / 2,
            f"{v:.4f} km²", va="center", fontsize=10.5, color="#1a1a1a")

ax.set_xlim(0, all_km2 * 1.28)
ax.set_xlabel("park area inside my 500 m zones (km²)", fontsize=9.5, color="#4a4a4a")
ax.grid(axis="x", color="#e8e8e8", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#cccccc")
ax.tick_params(length=0, labelsize=9.5)

ax.set_title(f"Dropping parks with no name loses {1 - named_km2 / all_km2:.0%} of the park "
             f"area you were measuring",
             fontsize=12, weight="bold", color="#1a1a1a", loc="left", pad=10)
plt.show()

**Kind two: nothing was missing until the join made it.** `sjoin` with `how="left"`
keeps every left-hand row whether or not it matched, and fills the right-hand columns with
missing values where it did not. Those nulls were not in either input file. The join created
them, and they mean "no match", which is information rather than absence.

The reason it matters is that the unmatched rows are **not a random sample**.

*What happens: two row counts, a null count, and two averages. Before you run it, decide whether the places your zones missed will look the same on average as the places they reached.*

In [ ]:
left_join = gpd.sjoin(class_places, zones, predicate="within", how="left")

print("class places going in:", len(class_places))
print("rows after how='left':", len(left_join))
print("rows with no zone    :", left_join["zone"].isna().sum())
print()

matched = left_join[left_join["zone"].notna()]
unmatched = left_join[left_join["zone"].isna()]
print("mean access_score, reached by a zone :", round(matched["access_score"].mean(), 3))
print("mean access_score, not reached       :", round(unmatched["access_score"].mean(), 3))

Neither count is 631, which catches almost everybody. `how="left"` keeps the 319
places that matched nothing **and** still duplicates the ones that matched twice, so a row count
stopped being a place count the moment the join ran.

The averages are the real finding. Places your zones reached average **0.894** on
`access_score`; the ones they missed average **0.836**. The gap is small but it is in one
direction, and it says the missingness carries information: the places further from your four
are also, on average, the ones students marked less accessible. Drop the unmatched rows and you
have quietly selected the more accessible half of campus and called it "the places near me".

**Ask of every missing value: is it missing for a reason connected to what I am measuring?** If
it is, dropping it is not cleaning, it is choosing an answer.

**Kind three: the geometry itself is missing.** This one is specific to `geopandas`,
and a missing shape is not the same as an empty one. The three-row table below is constructed
rather than loaded, because the course data has no such rows, and you want to have seen this
before the day you meet it.

*What happens: three rows and four ways of asking whether each has a shape. Before you run it, decide whether the empty polygon will count as missing.*

In [ ]:
from shapely.geometry import Point, Polygon

demo = gpd.GeoDataFrame(
    {"name": ["a real park", "no geometry at all", "an empty polygon"]},
    geometry=[Point(0, 0).buffer(10), None, Polygon()],
    crs=metres,
)

demo["is_na"] = demo.geometry.isna()        # True only for the missing one
demo["is_empty"] = demo.geometry.is_empty   # True only for the empty one
demo["area"] = demo.area                    # NaN vs 0.0: not the same thing

print(demo[["name", "is_na", "is_empty", "area"]].to_string(index=False))
print()
print("area.sum() over all three:", round(demo.area.sum(), 2), "-- the NaN was skipped silently")
print("rows after dropna(subset=['geometry']):", len(demo.dropna(subset=["geometry"])))

Read the last two lines carefully. `.area.sum()` **skipped the missing geometry
without a word** and returned the area of the other two, exactly as `pandas` skips `NA` in a
sum. If you had divided by `len(demo)` to get a mean area, you would have divided a two-row total
by three.

And the two failure modes need different tests: `.isna()` finds the missing shape and not the
empty one, `.is_empty` finds the empty one and not the missing one. An empty polygon has an area
of `0.0`, which is a real number and will average in as a zero. A missing one has `NaN`, which
will not. **Check both, and decide out loud which one a zero should be.**

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: what the missing rows were hiding

1. **Name the kind.** For each of these, say whether it is a missing attribute, a missing
   geometry, or a null the join manufactured, and say what you would do about it:
   *(a)* 369 parks with no `name`; *(b)* 319 class places with no `zone` after a left join;
   *(c)* a row whose `geometry` column holds `None`.
2. **Check your own answer from section 5.** The overlay against `buildings.parquet` reported a
   built share of the served area. `buildings` has nulls too. Print
   `buildings.isna().sum()`, then say whether any of those nulls could have changed that
   percentage, and why.

<details>
<summary><b>Check your answer</b></summary>

**1.** *(a)* A **missing attribute**. The polygon is fine, so do nothing to the rows: keep them
for anything geometric and only exclude them from a chart that needs labels. Never `.dropna()`
on `name` before measuring area, which is the 60 per cent mistake above. *(b)* A null the
**join manufactured** - it means "no match", and it was not in either input. Treat it as a
finding, and check whether the unmatched rows differ from the matched ones before you drop them,
which here they do. *(c)* A **missing geometry**. It cannot be measured, mapped or joined, so it
has to be found with `.isna()` on the geometry column and dealt with explicitly, because every
spatial method will silently return `False` or `NaN` for it.

**2.** `buildings` has nulls in `name` (7,749), `height` (8,320) and `building:levels` (7,245),
and **none of them can change the number**, because the overlay measured only `geometry`, and
the geometry column has no nulls at all. That is the point of the exercise: nulls only matter
when they are in a column the calculation actually touches. The section 5 result stands. Had you
been computing built **volume** from `height`, almost the entire file would have been unusable
and the honest answer would have been that this data cannot answer that question.

</details>

In [ ]:
# Your code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 7. The map, and the click test

**This is the hands-on part of the session.** A table is a claim. Click it, then take the two
Your Turns above as far as the time allows.

*What happens: an interactive map of your four zones, coloured by how many class places each contains. Before you run it, decide which zone you expect to be darkest.*

In [ ]:
zones_mapped = zones.merge(summary, on="zone")
zones_mapped.explore(column="class_places_within", cmap="YlOrRd",
                     tooltip=["zone", "class_places_within", "listings_within", "mean_access"],
                     style_kwds={"fillOpacity": 0.5})

*What happens: a static map of the class's places, with the ones inside your zones picked out. Before you run it, decide roughly what share of the 631 will be highlighted.*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

roads = gpd.read_parquet("../data/raw/roads.parquet").to_crs(metres)
roads.plot(ax=ax, color="#e8e8e8", linewidth=0.6)
class_places.plot(ax=ax, color="#bbbbbb", markersize=6)
zones.plot(ax=ax, facecolor="#e4572e", alpha=0.14, edgecolor="#e4572e")

inside = class_places[class_places["name"].isin(reached["name"])]
inside.plot(ax=ax, color="#e4572e", markersize=10)
places.plot(ax=ax, color="#003262", markersize=70, edgecolor="white")

ax.set_title("312 of the class's 631 places fall within 500 m of one of mine")
ax.set_axis_off()
minx, miny, maxx, maxy = zones.total_bounds
pad = 700
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + pad)
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Going deeper (optional)

### The rate that disagrees

Section 4 warned that `mean_access` treats a place listed once and a place listed 41 times as
equally important, and that the listing-weighted version would disagree with it. Here is the
other number, so the warning can be checked instead of believed.

Multiply each place's rate by the number of listings behind it, add those up per zone, and
divide by the listings. That is one extra column and one division.

*What happens: four zones with both rates side by side. Before you run it, decide whether the weighted numbers will spread further apart or closer together.*

In [ ]:
# how many of this place's listings said yes, rather than what share of them did
reached["yes_listings"] = reached["times_listed"] * reached["access_score"]

weighted = reached.groupby("zone").agg(
    yes_listings=("yes_listings", "sum"),
    all_listings=("times_listed", "sum"),
).reset_index()
weighted["weighted_access"] = (weighted["yes_listings"] / weighted["all_listings"]).round(3)

both = summary.merge(weighted[["zone", "weighted_access"]], on="zone")
both[["zone", "class_places_within", "mean_access", "weighted_access"]]

The two columns disagree, and the interesting part is *how*. The **ranking** survives:
Doe Library is still the most accessible corner and Games Of Berkeley still the least, under both
definitions. The **size of the difference** does not. Unweighted, the best and worst zones are
0.042 apart; weighted by listings they are 0.014 apart, which is close to nothing.

So the sentence "Doe Library's corner is noticeably more accessible than the others" is an
artefact of the definition, and the sentence "Doe Library's corner comes out on top" is not.
Those are very different claims, and only one of them survives asking the question a second way.

Neither number is wrong. `mean_access` asks "of the **places** near here, what share are
accessible", which lets a stairwell one student mentioned speak as loudly as a cafe forty-one
mentioned. `weighted_access` asks "of the **listings** near here, what share are of accessible
places". Whenever you average a column that is already an average, say out loud what the unit of
analysis is, and check whether your headline survives the other choice.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✏️ Your Turn

1. **Go further.** Change `predicate="within"` to `predicate="intersects"` in section 3 and
   re-run. For points strictly inside or strictly outside a zone the two agree, and the one case
   where they differ is a point lying exactly **on** the boundary, which `intersects` counts and
   `within` does not. Say in one sentence why that difference stops being a rare edge case the
   moment the left table holds polygons rather than points.

In [ ]:
# Your code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary

| Step | Tool |
|---|---|
| Ask a yes-or-no geometric question | `.within(shape)`, giving a mask |
| Attach the matched shape's columns | `gpd.sjoin(left, right, predicate="within")` |
| Keep unmatched rows too | `how="left"`, and the nulls it makes mean "no match" |
| Cut one layer by another | `gpd.overlay(a, b, how="intersection")` |
| Fuse overlapping shapes into one | `.dissolve()` |
| Count what is missing | `.isna().sum()`, per column |
| Never drop a row for a missing label before measuring its shape | check the column the maths touches |
| Missing shape against empty shape | `.isna()` and `.is_empty` catch different rows |
| Collapse the row explosion | `.groupby(...).agg(...)` |
| Count matched rows per group | `("name", "size")` |
| Check a claim by clicking it | `.explore(column=...)` |

Next: Week 9, where the numbers you have been computing become charts, and you find out how much
a chart can be made to lie.